# Cardinal: Bolide Infrasound
---
### Bolide characteristics were obtained from CNEOS database: https://cneos.jpl.nasa.gov/fireballs/
---
Outlines Cardinal processing pipeline 

1. Pre-processing: 

Read array data

Append array element location info

Convert units (if necessary)

Measure infrasound phase arrival times, ground-truth back azimuth, and source receiver distance (if origin time and location are provided)

Assess data quality

2. Processing:

Construct frequency bands using Segmentor

Apply Adaptive Array

Run Array Processor

Aggregate detections into families

3. Post-processing

Use families detections to compute signal/noise PSD's to measure peak frequency and frequency cutoffs

Use refined frequency cutoffs to measure SNR, peak amplitude, peak-to-peak amplitude, and beamform data

Use beam to compute spectrogram and scalogram

Cross-correlate array data using families detections and refined frequency cutoffs

In [ ]:
# Watermark notebook
import warnings, sys, cardinal, platform, os, logging, multiprocessing, psutil
#-----------------------------------------------------------------------------------------------------------------#
# Command to make all plots interactive
%matplotlib ipympl
#-----------------------------------------------------------------------------------------------------------------#
# Hide non-critical warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2" 
logging.getLogger('distributed.nanny').setLevel(logging.CRITICAL)
#-----------------------------------------------------------------------------------------------------------------#
# Import pac?kages as
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
#-----------------------------------------------------------------------------------------------------------------#
print(f"Python Platform: {platform.platform()}")
print(f"Tensorflow Version: {tf.__version__}")
print(f"Keras Version: {tf.keras.__version__}")
print()
print(f"Python {sys.version}")
print()
print(f"Number of cores available for processing: {multiprocessing.cpu_count()}")
print(f"Total memory available: {psutil.virtual_memory().total / 1e9, 'GB'}")
#-----------------------------------------------------------------------------------------------------------------#
# Specify client for parallel computing
# Good starting point is n_workers = number of cores / 2...make sure n_workers * memory_limit is less than total memory available
from dask.distributed import Client
n_workers = 4; client = Client(threads_per_worker=1, n_workers=n_workers, memory_limit='3GB')
#-----------------------------------------------------------------------------------------------------------------#
# Data directory
data_dir = 'Data/Bolide_Infrasound/'

## June 2, 2016 Bolide Recorded by I57US
---
Read data

In [ ]:
# Array and event time
array = 'I57US'; event_time = '2016-06-02T10:56:32'; source_lat = 33.8; source_lon = -110.9
#-----------------------------------------------------------------------------------------------------------------#
# Data and site files
data_filepath = data_dir+array+'/I57US.mseed'
array_coords_filepath = data_dir+array+'/is57.site'
#-----------------------------------------------------------------------------------------------------------------#
# Shift start and end times
shift_start=60*18.45; shift_end=60*48.45
#-----------------------------------------------------------------------------------------------------------------#
# Read data
st, st_filt, delay_times, GT_baz, distance = cardinal.plot_array_data(data_filepath, array_coords_filepath, event_time=event_time, source_lat=source_lat, source_lon=source_lon, 
                                                                      array=array, channel='BDF', trim_stream=[shift_start, shift_end], convert_units=[1/32527.3312], amp_units='Pressure [Pa]')

---
Data quality check

In [ ]:
# Manually reversing polarity at H2 (will automatically identify and fix)
for tr in st:
    if tr.stats.station == 'I57H2':
        tr.data *= -1
st_corr = cardinal.plot_data_quality(st, amp_units='Pressure [Pa]', reverse_polarities=True, return_stream=True)

---
Plot array coordinates

In [ ]:
ref_station = st_corr[0].stats.station
X, stnm = cardinal.get_array_coords(st_corr, ref_station, units='km')
cardinal.plot_array_coords(X, stnm, units='km')
plt.title(array+' Coordinates')

## 1. Segmentor
---
Third-octave spacing is best for detailed signal characterization

If you're planning on using the Adaptive Array, we suggest broadband frequency range (at least 3 orders of magnitude between f_min and f_max)

In [ ]:
f_bands = cardinal.make_custom_fbands(f_min=0.02, f_max=11, win_min=30, win_max=335.4, type='third_octave')
f_bands['fmax'].values[-1] = np.round(f_bands['fmax'].values[-1],0) # rounding so f_max becomes Nyquist
f_bands

## 2. Adaptive Array
---
Determine optimal subarray geometry for each frequency band

If batch processing or processing data in real-time, run the adaptive_array outside the processing block and use the second output to retrieve the subarray data for each new batch of data

In [ ]:
# Determine number of clusters to use for adaptive array
cardinal.element_pair_distances(st_corr, plot=True)

In [ ]:
%%time
_, subarrays_stnms = cardinal.adaptive_array(st_corr, f_bands, array_type='infrasound', plot=True, n_clusters=3) # vel is in km/s

---
Using output from adaptive_array to get subarray data

This keeps from needing to re-run the adaptive_array repeatedly - just use below code block inside your processing pipeline

In [ ]:
st_subarrays = cardinal.retrieve_subarray_data(st_corr, subarrays_stnms)

## 3. Array Processor
---
Process array data with and without the Adaptive Array to compare performance

In [ ]:
%%time 
# Without adaptive array
T, B, V, S = cardinal.sliding_time_array_fk_multifreq(st_corr, f_bands, client, signal_type='infrasound', adaptive_array=False)

In [ ]:
%%time 
# With adaptive array
T_adaptive, B_adaptive, V_adaptive, S_adaptive = cardinal.sliding_time_array_fk_multifreq(st_subarrays, f_bands, client, signal_type='infrasound', adaptive_array=True)

---
Plot results

In [ ]:
# without adaptive array
cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T, B, V, S, semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + '\n' + str(distance) + ' [km]')

In [ ]:
# with adaptive array
cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], normalize=True,
                                       bandpass=[0.5,5], GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + ' - Adaptive Array\n' + str(distance) + ' [km]')

## 4. Aggregator
---
Isolating each of the 3 distinct arrivals

In [ ]:
%%time
ref_time = st[0].stats.starttime.matplotlib_date
ix, pixels_in_families, families = cardinal.make_families(T_adaptive, B_adaptive, V_adaptive, S_adaptive, f_bands, ref_time,
                                                          dist_threshold=1, min_pixels=100, sigma_t=2, sigma_f=2, sigma_b=10, p_threshold=0.075, 
                                                          family_grouping='kdtree')

In [ ]:
cardinal.df_families(ref_time, families)

---
Plot results with Aggregator

In [ ]:
cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]',
                                       pixels_in_families=pixels_in_families, ix=ix, 
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Compute PSD

In [ ]:
cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]',
                                       pixels_in_families=pixels_in_families, ix=ix, families=families,
                                       compute_metrics=True, family_idx=1, trim_family_window=[20,80],
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' km')

---
Beamform data

In [ ]:
st_beam = cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                                 clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.25,8.25], normalize=True,
                                                 GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]',
                                                 pixels_in_families=pixels_in_families, ix=ix, families=families,
                                                 compute_metrics=True, family_idx=1, trim_family_window=[20,80],
                                                 beamform_data=True, return_beam=True,
                                                 title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Plot spectrogram

In [ ]:
cardinal.plot_spectrogram(st_beam, element='Beam', bandpass=[0.25,8.25], t_lim=[400,900], v_lim=[-10,2], delay_times=delay_times, amp_units='Pressure [Pa]', normalize=True, title=array)

---
Plot scalogram

In [ ]:
%%time
cardinal.plot_scalogram(st_beam, element='Beam', bandpass=[0.25,8.25], trim_stream=[400,900], v_lim=[-10,2], f_lim=[0.25,10], delay_times=delay_times, normalize=True, amp_units='Pressure [Pa]', title=array)

---
Plot cross-correlation

In [ ]:
%%time
cardinal.plot_cross_correlation(st_corr, f_bands, bandpass=[0.25,8.25], delay_times=delay_times, families=families, family_idx=1, trim_family_window=[20,80], title1=array, title2=array)

## May 19, 2019 Bolide Recorded by I07AU
---
Read data

In [ ]:
# Array and event time
array = 'I07AU'; event_time = '2019-05-19T14:47:03'; source_lat = -23.6; source_lon = 132.8
#-----------------------------------------------------------------------------------------------------------------#
# Data and site files
data_filepath = data_dir+array+'/I07AU.mseed'
array_coords_filepath = data_dir+array+'/is07.site'
#-----------------------------------------------------------------------------------------------------------------#
# Shift start and end times
shift_start=60*12.95; shift_end=60*42.95
#-----------------------------------------------------------------------------------------------------------------#
# Read data
st, st_filt, delay_times, GT_baz, distance = cardinal.plot_array_data(data_filepath, array_coords_filepath, event_time=event_time, source_lat=source_lat, source_lon=source_lon, 
                                                                      array=array, channel='BDF', trim_stream=[shift_start, shift_end], convert_units=[1/6971], amp_units='Pressure [Pa]')

---
Data quality check

In [ ]:
cardinal.plot_data_quality(st,  amp_units='Pressure [Pa]')

---
Plot array coordinates

In [ ]:
ref_station = st[0].stats.station
X, stnm = cardinal.get_array_coords(st, ref_station, units='km')
cardinal.plot_array_coords(X, stnm, units='km')
plt.title(array+' Coordinates')

## 1. Segmentor
---

In [ ]:
f_bands = cardinal.make_custom_fbands(f_min=0.02, f_max=11, win_min=30, win_max=335.4, type='third_octave')
f_bands['fmax'].values[-1] = np.round(f_bands['fmax'].values[-1],0) # rounding so f_max becomes Nyquist
f_bands

## 2. Adaptive Array
---

In [ ]:
# Determine number of clusters to use for adaptive array
cardinal.element_pair_distances(st, plot=True)

In [ ]:
%%time
st_subarrays, _ = cardinal.adaptive_array(st, f_bands, array_type='infrasound', plot=True, n_clusters=4)

## 3. Array Processor
---

In [ ]:
%%time 
# Without adaptive array
T, B, V, S = cardinal.sliding_time_array_fk_multifreq(st, f_bands, client, signal_type='infrasound', adaptive_array=False)

In [ ]:
%%time 
# With adaptive array
T_adaptive, B_adaptive, V_adaptive, S_adaptive = cardinal.sliding_time_array_fk_multifreq(st_subarrays, f_bands, client, signal_type='infrasound', adaptive_array=True)

---
Plot results

In [ ]:
# without adaptive array
cardinal.plot_sliding_window_multifreq(st, f_bands, T, B, V, S, 
                                       semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + '\n' + str(distance) + ' [km]')

In [ ]:
# with adaptive array
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + ' - Adaptive Array\n' + str(distance) + ' [km]')

## 4. Aggregator
---

In [ ]:
%%time
ref_time = st[0].stats.starttime.matplotlib_date
ix, pixels_in_families, families = cardinal.make_families(T_adaptive, B_adaptive, V_adaptive, S_adaptive, f_bands, ref_time,
                                                          dist_threshold=3, min_pixels=200, sigma_t=2, sigma_f=2, sigma_b=2, p_threshold=0.2, 
                                                          family_grouping='kdtree')

In [ ]:
cardinal.df_families(ref_time, families)

---
Plot results with Aggregator

In [ ]:
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], 
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', normalize=True,
                                       pixels_in_families=pixels_in_families, ix=ix, 
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Compute PSD

In [ ]:
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], 
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]',
                                       pixels_in_families=pixels_in_families, ix=ix, families=families, 
                                       compute_metrics=True, noise_start=8.5, family_idx=0, trim_family_window=[50,100],
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' km')

---
Beamform data

In [ ]:
st_beam = cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                                 clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.25,3.25], 
                                                 GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]',
                                                 pixels_in_families=pixels_in_families, ix=ix, families=families, 
                                                 compute_metrics=True, noise_start=8.5, family_idx=0, trim_family_window=[50,100],
                                                 beamform_data=True, return_beam=True,
                                                 title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Plot spectrogram

In [ ]:
cardinal.plot_spectrogram(st_beam, element='Beam', bandpass=[0.25,3.25], t_lim=[400,850], v_lim=[-10,2], delay_times=delay_times, normalize=True, amp_units='Pressure [Pa]', title=array)

---
Plot scalogram

In [ ]:
%%time
cardinal.plot_scalogram(st_beam, element='Beam', bandpass=[0.25,3.25], trim_stream=[400,850], v_lim=[-10,2], f_lim=[0.25,10], delay_times=delay_times, normalize=True, amp_units='Pressure [Pa]', title=array)

---
Plot cross-correlation

In [ ]:
%%time
cardinal.plot_cross_correlation(st, f_bands, bandpass=[0.25,3.25], delay_times=delay_times, families=families, family_idx=0, trim_family_window=[50,100], title1=array, title2=array)

## February 26, 2015 Bolide Recorded by I53US
---
Read data

In [ ]:
# Array and event time
array = 'I53US'; event_time = '2015-02-26T22:06:24'; source_lat = 68; source_lon = -149
#-----------------------------------------------------------------------------------------------------------------#
# Data and site files
data_filepath = data_dir+array+'/I53US.mseed'
array_coords_filepath = data_dir+array+'/is53.site'
#-----------------------------------------------------------------------------------------------------------------#
# Shift start and end times
shift_start=60*13.6; shift_end=60*33.6
#-----------------------------------------------------------------------------------------------------------------#
# Read data
st, st_filt, delay_times, GT_baz, distance = cardinal.plot_array_data(data_filepath, array_coords_filepath, event_time=event_time, source_lat=source_lat, source_lon=source_lon, 
                                                                      array=array, channel='BDF', trim_stream=[shift_start, shift_end], convert_units=[1/244032], amp_units='Pressure (Pa)')

---
Data quality check

In [ ]:
cardinal.plot_data_quality(st, amp_units='Pressure [Pa]')

---
Plot array coordinates

In [ ]:
ref_station = st[0].stats.station
X, stnm = cardinal.get_array_coords(st, ref_station, units='km')
cardinal.plot_array_coords(X, stnm, units='km')
plt.title(array+' Array Coordinates')

## 1. Segmentor
---

In [ ]:
f_bands = cardinal.make_custom_fbands(f_min=0.02, f_max=11, win_min=30, win_max=335.4, type='third_octave')
f_bands['fmax'].values[-1] = np.round(f_bands['fmax'].values[-1],0) # rounding so f_max becomes Nyquist
f_bands

## 2. Adaptive Array
---

In [ ]:
# Determine number of clusters to use for adaptive array
cardinal.element_pair_distances(st, plot=True)

In [ ]:
%%time
st_subarrays, _ = cardinal.adaptive_array(st, f_bands, array_type='infrasound', plot=True, n_clusters=6)

## 3. Array Processor
---

In [ ]:
%%time 
# Without adaptive array
T, B, V, S = cardinal.sliding_time_array_fk_multifreq(st, f_bands, client, signal_type='infrasound', adaptive_array=False)

In [ ]:
%%time 
# With adaptive array
T_adaptive, B_adaptive, V_adaptive, S_adaptive = cardinal.sliding_time_array_fk_multifreq(st_subarrays, f_bands, client, signal_type='infrasound', adaptive_array=True)

---
Plot results

In [ ]:
# without adaptive array
cardinal.plot_sliding_window_multifreq(st, f_bands, T, B, V, S, 
                                       semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + '\n' + str(distance) + ' [km]')

In [ ]:
# with adaptive array
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       semblance_threshold=0.5, clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], normalize=True,
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', title=array + ' - Adaptive Array\n' + str(distance) + ' [km]')

## 4. Aggregator
---

In [ ]:
%%time
ref_time = st[0].stats.starttime.matplotlib_date
ix, pixels_in_families, families = cardinal.make_families(T_adaptive, B_adaptive, V_adaptive, S_adaptive, f_bands, ref_time,
                                                          dist_threshold=2, min_pixels=100, sigma_t=2, sigma_f=2, sigma_b=5, p_threshold=0.4, 
                                                          family_grouping='kdtree')

In [ ]:
cardinal.df_families(ref_time, families)

---
Plot results with Aggregator

In [ ]:
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], 
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', normalize=True, 
                                       pixels_in_families=pixels_in_families, ix=ix, 
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Compute PSD

In [ ]:
cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.5,5], 
                                       GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', normalize=True, 
                                       pixels_in_families=pixels_in_families, ix=ix, families=families,
                                       compute_metrics=True, trim_family_window=[75,125], noise_start=5,
                                       title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' km')

---
Beamform data

In [ ]:
st_beam = cardinal.plot_sliding_window_multifreq(st, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                                 clim_vtr=[0.2,0.45], clim_baz=[-90,90], t_lim=[0,shift_end-shift_start], bandpass=[0.1,8], 
                                                 GT_baz=GT_baz, delay_times=delay_times, amp_units='Pressure [Pa]', normalize=True, 
                                                 pixels_in_families=pixels_in_families, ix=ix, families=families,
                                                 compute_metrics=True, trim_family_window=[75,125], noise_start=5,
                                                 beamform_data=True, return_beam=True,
                                                 title=array + ' - Adaptive Array - Aggregator\n' + str(distance) + ' [km]')

---
Plot spectrogram

In [ ]:
cardinal.plot_spectrogram(st_beam, element='Beam', bandpass=[0.1,8], t_lim=[100,700], v_lim=[-10,2], delay_times=delay_times, normalize=True, amp_units='Pressure [Pa]', title=array)

---
Plot scalogram

In [ ]:
%%time
cardinal.plot_scalogram(st_beam, element='Beam', bandpass=[0.1,8], trim_stream=[100,700], v_lim=[-10,2], f_lim=[0.25,10], delay_times=delay_times, normalize=True, amp_units='Pressure [Pa]', title=array)

---
Plot cross-correlation

In [ ]:
%%time
cardinal.plot_cross_correlation(st, f_bands, bandpass=[0.1,8], delay_times=delay_times, families=families, trim_family_window=[75,125], title1=array, title2=array)